# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
import os
os.environ["JAVA_HOME"] = "/home/trar3243/miniforge3/envs/spark_env"

from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

from pathlib import Path

checkpoint_dir = Path("./spark_checkpoint").resolve() # I like to checkpoint rather than cache because I like having it on disk 
checkpoint_dir.mkdir(parents=True, exist_ok=True)

sc.setCheckpointDir(checkpoint_dir.as_uri())

print(sc.getCheckpointDir())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 16:51:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


file:/home/trar3243/repos/datacenter_scale/lab4-pyspark-patent/spark_checkpoint/4e1cc66c-5e34-4ad9-a0d3-957c67768a3f


Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

Goal: Get am RDD which has citing patents alongside cited patent IDs 

Step 1: get the RDDs to not have single string for each record. We split by commas for both RDDs. We also filter out "empty" postates, because we do not want to include them. 
step 2: key the patents table by its patent, and key the citations table by citing. This will be used for joining the two. Include postate for the patent table, and include the cited from the citations table (for later use)
step 3: join the patents and citations table on patents.patent=citations.citing. Now we have an RDD where each record corresponds to as many citations as the patent makes. 
step 4: checkpoint the resulting RDD. At this point, we have an RDD with as many records per patent as there are citations. 

In [6]:


split_rdd_patents = rddPatents.map(lambda row: row.split(',')).filter(lambda row: row[5] != '""') # split with comma separated still has the header , also filter out nan states 
split_rdd_citations = rddCitations.map(lambda row: row.split(','))

rdd_patents_keyed_by_patent = split_rdd_patents.map(lambda row: (row[0], row[5])) # get patent as key 
rdd_citations_keyed_by_citing = split_rdd_citations.map(lambda row: (row[0], row[1])) # get citing as key 
# rdd_citations_keyed_by_cited = split_rdd_citations.map(lambda row: (row[1], row[0])) # get cited as key 

patents_joined_by_citing = rdd_patents_keyed_by_patent.join(rdd_citations_keyed_by_citing) # keyed by citing 
patents_joined_by_citing.checkpoint()
patents_joined_by_citing.take(10) # forces the checkpoint to write to disk 


[('3858607', ('"CA"', '2682386')),
 ('3858607', ('"CA"', '2897836')),
 ('3858607', ('"CA"', '2904068')),
 ('3858607', ('"CA"', '2969218')),
 ('3858607', ('"CA"', '2980139')),
 ('3858607', ('"CA"', '3038499')),
 ('3858607', ('"CA"', '3126915')),
 ('3858607', ('"CA"', '3272218')),
 ('3858607', ('"CA"', '3331583')),
 ('3858607', ('"CA"', '3415282'))]

Goal: Filter the previous result to include only records for each patent where the patent cites an in-state patent 

Step 1: create an RDD with the postate and cited fields as the key. This will be used for joining back to patents table. Payload is the "citing" records which WAS part of our key before. 
Step 2: Join that RDD back with patents on POSTATE and "cited". 
Step 3: Checkpoint result. We now have an RDD with as many records per patent as there are in-state "citing"s in that patent. 

In [7]:
patents_joined_by_citing_keyed_by_postate_cited = patents_joined_by_citing.map(lambda row: (
            (row[1][0], row[1][1]), # postate, cited
            row[0]# citing 
        )
    )

rdd_patents_keyed_by_postate_patent = split_rdd_patents.map(lambda row: ((row[5], row[0]), row)) # get postate and patent as key 

full = patents_joined_by_citing_keyed_by_postate_cited.join(rdd_patents_keyed_by_postate_patent) # keyed by postate and cited with value for citing, then cited row 
full.checkpoint()
full.take(5)

[(('"PA"', '4111772'),
  ('4289597',
   ['4111772',
    '1978',
    '6822',
    '1976',
    '"US"',
    '"PA"',
    '442270',
    '2',
    '22',
    '205',
    '1',
    '19',
    '3',
    '9',
    '1',
    '0.6667',
    '0.4444',
    '11',
    '5.6667',
    '0.3333',
    '0.3333',
    '0',
    '0'])),
 (('"PA"', '4111772'),
  ('4549946',
   ['4111772',
    '1978',
    '6822',
    '1976',
    '"US"',
    '"PA"',
    '442270',
    '2',
    '22',
    '205',
    '1',
    '19',
    '3',
    '9',
    '1',
    '0.6667',
    '0.4444',
    '11',
    '5.6667',
    '0.3333',
    '0.3333',
    '0',
    '0'])),
 (('"TX"', '4511842'),
  ('5168234',
   ['4511842',
    '1985',
    '9237',
    '1981',
    '"US"',
    '"TX"',
    '497495',
    '2',
    '7',
    '324',
    '4',
    '43',
    '2',
    '27',
    '1',
    '0',
    '0.5',
    '6.5185',
    '10.5',
    '0.5',
    '0.5',
    '0.4815',
    '0.4815'])),
 (('"TX"', '4511842'),
  ('4968940',
   ['4511842',
    '1985',
    '9237',
    '1981',
    '

Goal: Essentially group by the citing patent, getting the counts of corresponding records (in-state citings) for each of the patent

Step 1: key the RDD by citing. This is THE "patent ID". The payload is the literal "1". This will be used to sum to get the count
Step 2: Reduce that RDD by key. The key is the "citing" key. The reduction essentially acts as a for-loop for each unique patent key, iteratively summing the payload (all 1s)
Step 3: order by the count
Step 4: checkpoint. At this point we have the "patent ID" alongside the count of in-state citings. 

In [8]:
keyed_by_citing = full.map(lambda row: (row[1][0], 1))# get the citing as the key again , but the value is the literal 1. We will use this to count 

counts_by_citing = keyed_by_citing.reduceByKey( # reduce by key will execute this operation for each unique key set. The operation is summation of the 1 literals 
        lambda a, b: a + b
    ).sortBy(lambda x: x[1], ascending=False) # sort by value descending 

counts_by_citing.checkpoint()

counts_by_citing.take(10)

[('5959466', 125),
 ('5983822', 103),
 ('6008204', 100),
 ('5952345', 98),
 ('5998655', 96),
 ('5958954', 96),
 ('5936426', 94),
 ('5739256', 90),
 ('5978329', 90),
 ('5913855', 90)]

Goal: Join the "count" RDD from prior step back to the original patent RDD. This will let us get each patent's full information, alongside its in-state citings count 
Step 1: join by the "citing" patent key 
Step 2: checkpoint 

In [9]:
# now join back to the main 
result = split_rdd_patents.map(lambda row: (row[0], row)).join(counts_by_citing)
result.checkpoint()
result.take(10)

[('3869999',
  (['3869999',
    '1975',
    '5548',
    '1973',
    '"US"',
    '"NY"',
    '',
    '1',
    '10',
    '112',
    '6',
    '63',
    '8',
    '1',
    '0.25',
    '0',
    '0',
    '23',
    '35.625',
    '',
    '',
    '',
    ''],
   1)),
 ('3881410',
  (['3881410',
    '1975',
    '5604',
    '1973',
    '"US"',
    '"IL"',
    '409245',
    '2',
    '4',
    '101',
    '6',
    '69',
    '7',
    '7',
    '0.4286',
    '0.6122',
    '0.4444',
    '8.1429',
    '13.1429',
    '0',
    '0',
    '0.5',
    '0.4286'],
   1)),
 ('3995406',
  (['3995406',
    '1976',
    '6185',
    '1975',
    '"US"',
    '"CA"',
    '',
    '1',
    '33',
    '411',
    '5',
    '59',
    '7',
    '7',
    '0.2857',
    '0.4898',
    '0',
    '9.1429',
    '33.1429',
    '',
    '',
    '',
    ''],
   2)),
 ('4023627',
  (['4023627',
    '1977',
    '6346',
    '1975',
    '"US"',
    '"PA"',
    '272755',
    '2',
    '8',
    '173',
    '5',
    '51',
    '5',
    '2',
    '0.8',
  

Goal: Clean up the result to look more like what we want. 
Step 1: Flatten the "row" element with the count field. This makes all the fields look the same. Also get rid of the redundant key. 
Step 2: order by the "in-state count" descinding 
Step 3: convert back into CSV format 
Step 4: print out. 

In [10]:
flattened_result = result.map(lambda x: [*x[1][0], x[1][1]]) # this is the row, count. The * is a splat operator, and it unpacks the objects . gets rid of the redundant "citing" key 
ordered_results = flattened_result.sortBy(lambda x: x[-1], ascending=False) # order by the last 
csv_rdd = ordered_results.map(lambda row: ",".join(str(item) for item in row)) # convert back to CSV string 
csv_rdd.checkpoint()

csv_rdd.take(10)

['5959466,1999,14515,1997,"US","CA",5310,2,,326,4,46,159,0,1,,0.6186,,4.8868,0.0455,0.044,,,125',
 '5983822,1999,14564,1998,"US","TX",569900,2,,114,5,55,200,0,0.995,,0.7201,,12.45,0,0,,,103',
 '6008204,1999,14606,1998,"US","CA",749584,2,,514,3,31,121,0,1,,0.7415,,5,0.0085,0.0083,,,100',
 '5952345,1999,14501,1997,"US","CA",749584,2,,514,3,31,118,0,1,,0.7442,,5.1102,0,0,,,98',
 '5958954,1999,14515,1997,"US","CA",749584,2,,514,3,31,116,0,1,,0.7397,,5.181,0,0,,,96',
 '5998655,1999,14585,1998,"US","CA",,1,,560,1,14,114,0,1,,0.7387,,5.1667,,,,,96',
 '5936426,1999,14466,1997,"US","CA",5310,2,,326,4,46,178,0,1,,0.58,,11.2303,0.0765,0.073,,,94',
 '5739256,1998,13983,1995,"US","CA",70060,2,15,528,1,15,453,0,1,,0.8232,,15.1104,0.1124,0.1082,,,90',
 '5925042,1999,14445,1997,"US","CA",733846,2,,606,3,32,242,0,1,,0.7382,,8.3471,0,0,,,90',
 '5913855,1999,14417,1997,"US","CA",733846,2,,606,3,32,242,0,1,,0.7403,,8.3595,0,0,,,90']